# Mode Selection
Set finetuning = 1 if you want to finetune, or set finetuning = 0 if you want to just load the saved model to get inference.

In [ ]:
finetuning = 0
load_and_run = not finetuning

# Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Load Data

In [ ]:
import pandas as pd

data = pd.read_csv('/content/drive/MyDrive/#Research/# GB/train_test_sarcasm_data.csv')
data.dropna(inplace = True)
data

In [ ]:
test = data[data.test==1]
test

# Prediction of Test Set


In [ ]:
# Use a pipeline as a high-level helper
import torch
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("raquiba/sarcasm-detection-BanglaSARC")
model = AutoModelForSequenceClassification.from_pretrained("raquiba/sarcasm-detection-BanglaSARC")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device)
model.to(device)
model.eval()



male_preds = []
for sentence in tqdm(test['male']):
  inputs = tokenizer(sentence, return_tensors="pt", max_length= 512, truncation=True)
  with torch.no_grad():
      logits = model(**inputs.to(device)).logits
  predicted_class_id = logits.argmax().item()
  male_preds.append(predicted_class_id)

female_preds = []
for sentence in tqdm(test['female']):
  inputs = tokenizer(sentence, return_tensors="pt", max_length= 512, truncation=True)

  with torch.no_grad():
      logits = model(**inputs.to(device)).logits


  predicted_class_id = logits.argmax().item()
  female_preds.append(predicted_class_id)
test['male_prediction'] = male_preds
test['female_prediction'] = female_preds

In [ ]:
test

In [ ]:
len(test[test['male_prediction'] != test['female_prediction']])

In [ ]:
#for male
x = sum(test['_original_label'] != test['male_prediction'])
print('Total data: ', len(test),'Total mismatch: ', x,'Accuracy: ', 1- (x/len(test)) )

In [ ]:
#for female
x = sum(test['_original_label'] != test['female_prediction'])
print('Total data: ', len(test),'Total mismatch: ', x,'Accuracy: ', 1- (x/len(test)) )

In [ ]:
test.to_csv('/content/drive/MyDrive/#Research/# GB/1.revision_zs_sarcasm_result.csv', index = False)

# Result for multiple run

In [ ]:
import pandas as pd
test = pd.read_csv('/content/drive/MyDrive/#Research/# GB/1.revision_zs_sarcasm_result.csv')
import numpy as np

def bootstrap_ci(y_true, y_pred, B=500):
    """Compute bootstrap confidence interval for accuracy."""
    N = len(y_true)
    original_acc = np.mean(y_pred == y_true)

    bootstrap_accs = []

    for _ in range(B):
        indices = np.random.choice(N, N, replace=True)  # Sample with replacement
        y_sample = y_true[indices]
        acc = np.mean(y_pred[indices] == y_sample)
        bootstrap_accs.append(acc)
    # Compute 95% confidence interval
    lower, upper = np.percentile(bootstrap_accs, [2.5, 97.5])

    return original_acc, (lower, upper), np.mean(bootstrap_accs)


y_true = test['_original_label'].values  # Ground truth labels
y_pred = test['male_prediction'].values
acc, ci, ma = bootstrap_ci(y_true, y_pred)
print('Male: ')
print(f"Accuracy: {acc:.4f}")
print(f"95% Confidence Interval: {ci}")
print(f"mean accuracy: {ma}")

y_true = test['_original_label'].values
y_pred = test['female_prediction'].values
acc, ci, ma = bootstrap_ci(y_true, y_pred)
print('\nFemale: ')
print(f"Accuracy: {acc:.4f}")
print(f"95% Confidence Interval: {ci}")
print(f"mean accuracy: {ma}")

In [ ]:
# Statistical Parity Difference (SPD) and Equal Opportunity Difference (EOD) Calculation
import numpy as np
import pandas as pd
np.random.seed(0)

test = pd.read_csv('1.revision_zs_sarcasm_result.csv')


# Function to calculate Statistical Parity Difference (SPD)
def calculate_spd(male_pred, female_pred):
    # Calculate probabilities of positive predictions for males and females
    p_male = np.mean(male_pred == 1)  # Proportion of positive predictions for males
    p_female = np.mean(female_pred == 1)  # Proportion of positive predictions for females
    
    # Return the absolute value of the Statistical Parity Difference
    return np.abs(p_male - p_female)

# Function to calculate Equal Opportunity Difference (EOD)
def calculate_eod(male_pred, female_pred, y_true):
    # True Positive Rate for males
    tpr_male = np.mean((male_pred == 1) & (y_true == 1))  # True positives for males
    
    # True Positive Rate for females
    tpr_female = np.mean((female_pred == 1) & (y_true == 1))  # True positives for females
    
    # Return the absolute value of the Equal Opportunity Difference
    return np.abs(tpr_male - tpr_female)

# Function to calculate bootstrap confidence intervals for SPD and EOD
def bootstrap_ci_spd_eod(male_pred, female_pred, y_true, n_iterations=500, ci=95):
    n = len(male_pred)
    
    # Arrays to store SPD and EOD for each bootstrap sample
    spd_values = np.zeros(n_iterations)
    eod_values = np.zeros(n_iterations)
    
    # Perform bootstrap sampling
    for i in range(n_iterations):
        # Resample with replacement
        sample_indices = np.random.choice(n, size=n, replace=True)
        male_pred_resampled = male_pred[sample_indices]
        female_pred_resampled = female_pred[sample_indices]
        y_true_resampled = y_true[sample_indices]
        
        # Calculate SPD and EOD for the resampled data
        spd_values[i] = calculate_spd(male_pred_resampled, female_pred_resampled)
        eod_values[i] = calculate_eod(male_pred_resampled, female_pred_resampled, y_true_resampled)
    
    # Calculate the mean of SPD and EOD
    spd_mean = np.mean(spd_values)
    eod_mean = np.mean(eod_values)
    
    # Calculate the confidence interval bounds for SPD and EOD
    lower_percentile = (100 - ci) / 2
    upper_percentile = 100 - lower_percentile
    spd_lower_bound = np.percentile(spd_values, lower_percentile)
    spd_upper_bound = np.percentile(spd_values, upper_percentile)
    eod_lower_bound = np.percentile(eod_values, lower_percentile)
    eod_upper_bound = np.percentile(eod_values, upper_percentile)
    
    return (np.abs(spd_mean), np.abs(spd_lower_bound), np.abs(spd_upper_bound)), (np.abs(eod_mean), np.abs(eod_lower_bound), np.abs(eod_upper_bound))


y_true = test._original_label.values
male_pred = test.male_prediction.values
female_pred = test.female_prediction.values

# Calculate bootstrap CI and mean for SPD and EOD
(spd_mean, spd_lower, spd_upper), (eod_mean, eod_lower, eod_upper) = bootstrap_ci_spd_eod(
    male_pred, female_pred, y_true, n_iterations=500, ci=95
)

print(f"SPD 95% Confidence Interval: [{spd_lower:.3f}, {spd_upper:.3f}]\tSPD Mean: {spd_mean:.3f}")
print('---------------------------------------------------------------------------------------------')
print(f"EOD 95% Confidence Interval: [{eod_lower:.3f}, {eod_upper:.3f}]\tEOD Mean: {eod_mean:.3f}")